In [1]:
from load_data import load_raw_data
from clean_data import clean_data
from split_data import split_data
from evaluate_model import evaluate_model
from config import RANDOM_STATE
from build_pipeline import objective,build_pipeline
from feature_engineering import build_new_features
import optuna
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

In [3]:
# Loading raw data
dataset=load_raw_data()

2026-08-20 14:03:46,251 | INFO | load_data | Raw dataset loaded.


In [4]:
# Cleaning data
dataset_cleaned=clean_data(dataset)

2026-08-20 14:03:46,259 | INFO | clean_data | Dropped customerID.
2026-08-20 14:03:46,262 | INFO | clean_data | Converted TotalCharges to numeric.
2026-08-20 14:03:46,268 | INFO | clean_data | Rows containing NaN before cleaning: 11
2026-08-20 14:03:46,278 | INFO | clean_data | Removed 11 rows with invalid TotalCharges. Remaining rows with NaN: 0.


In [5]:
# Add new features
#dataset_cleaned=build_new_features(dataset_cleaned)

In [6]:
# Splitting data
X_train, X_test, y_train, y_test=split_data(dataset_cleaned, random_state=RANDOM_STATE)

2026-08-20 14:03:46,484 | INFO | split_data | Data split into train/test set with 0.8/0.2 proportion and random_state=0.


### XGBoost

In [6]:
# Creating optuna study for XGBoost
study_XGB = optuna.create_study(direction="maximize")

[I 2026-08-11 10:26:48,276] A new study created in memory with name: no-name-1372aacd-0560-47c4-81c9-da069ce78e7b


In [7]:
# Optimizing optuna study for XGBoost
study_XGB.optimize(lambda trial: objective(trial, X_train, y_train, model='XGBoost'),n_trials=100)

[I 2026-08-11 10:26:51,494] Trial 0 finished with value: 0.5544260451878306 and parameters: {'xgb_learning_rate': 0.0012184638966534816, 'xgb_max_depth': 4, 'xgb_min_child_weight': 17.493896605197982, 'xgb_gamma': 6.151551363875285, 'xgb_subsample': 0.9044295164555402, 'xgb_colsample_bytree': 0.9560569484501648, 'xgb_colsample_bylevel': 0.8264545133551691, 'xgb_reg_alpha': 2.4250199566746327, 'xgb_reg_lambda': 1.870537046146241e-08, 'xgb_scale_pos_weight': 6.564005741873099}. Best is trial 0 with value: 0.5544260451878306.
[I 2026-08-11 10:26:53,100] Trial 1 finished with value: 0.6332124479080132 and parameters: {'xgb_learning_rate': 0.0181980452105263, 'xgb_max_depth': 6, 'xgb_min_child_weight': 19.21752116818887, 'xgb_gamma': 9.409010653016022, 'xgb_subsample': 0.6520110432187467, 'xgb_colsample_bytree': 0.8197856337356415, 'xgb_colsample_bylevel': 0.5395694064519774, 'xgb_reg_alpha': 0.030694292410767616, 'xgb_reg_lambda': 7.537531236033466e-07, 'xgb_scale_pos_weight': 2.1908438281

In [8]:
print("Best score for XGB:", study_XGB.best_value)
print("Best params for XGB:", study_XGB.best_params)

Best score for XGB: 0.6387903984647723
Best params for XGB: {'xgb_learning_rate': 0.027819002320813163, 'xgb_max_depth': 3, 'xgb_min_child_weight': 17.146484277484138, 'xgb_gamma': 8.718461351247017, 'xgb_subsample': 0.8310982031645213, 'xgb_colsample_bytree': 0.7488481655554002, 'xgb_colsample_bylevel': 0.6075111289796862, 'xgb_reg_alpha': 0.00021554204480234767, 'xgb_reg_lambda': 11.361424758522615, 'xgb_scale_pos_weight': 1.9472643751544718}


In [9]:
# Fitting Logistic Regression pipeline with the best trial
pipe_XGB=build_pipeline(trial=study_XGB.best_trial, model= 'XGBoost')
fitted_pipe_XGB=pipe_XGB.fit(X_train, y_train)

### Logistic Regression

In [7]:
# Creating optuna study for Logistic Regression
study_LR = optuna.create_study(direction="maximize")

[I 2026-08-20 14:04:02,063] A new study created in memory with name: no-name-0ca86de1-c474-4c6e-95c2-7b3cfa3d1dbf


In [8]:
# Optimizing optuna study for Logistic Regression
study_LR.optimize(lambda trial: objective(trial, X_train, y_train, model='Logistic Regression'),n_trials=100)

[I 2026-08-20 14:04:06,857] Trial 0 finished with value: 0.6316019723712125 and parameters: {'solver': 'lbfgs', 'C': 513.622487937945}. Best is trial 0 with value: 0.6316019723712125.
[I 2026-08-20 14:04:08,015] Trial 1 finished with value: 0.6289045437275108 and parameters: {'solver': 'newton-cg', 'C': 0.05950437617664868}. Best is trial 0 with value: 0.6316019723712125.
[I 2026-08-20 14:04:09,358] Trial 2 finished with value: 0.6278173341076503 and parameters: {'solver': 'liblinear', 'C': 0.010870019509436018}. Best is trial 0 with value: 0.6316019723712125.
[I 2026-08-20 14:04:10,898] Trial 3 finished with value: 0.6314744355502087 and parameters: {'solver': 'liblinear', 'C': 7.67266893880213}. Best is trial 0 with value: 0.6316019723712125.
[I 2026-08-20 14:04:10,995] Trial 4 finished with value: 0.6259313697672504 and parameters: {'solver': 'liblinear', 'C': 0.0046288062951223945}. Best is trial 0 with value: 0.6316019723712125.
[I 2026-08-20 14:04:11,086] Trial 5 finished with va

In [9]:
print("Best score for LR:", study_LR.best_value)
print("Best params for LR:", study_LR.best_params)

Best score for LR: 0.6324543015853048
Best params for LR: {'solver': 'saga', 'C': 37.089194536771}


In [11]:
# Fitting Logistic Regression pipeline with the best trial
pipe_LR=build_pipeline(trial=study_LR.best_trial, model= 'Logistic Regression')
fitted_pipe_LR=pipe_LR.fit(X_train, y_train)

### K-Nearest Neighbors

In [12]:
# Creating optuna study for K-Nearest Neighbors
study_KNN = optuna.create_study(direction="maximize")

[I 2026-06-12 23:58:14,203] A new study created in memory with name: no-name-c86b3ddf-161a-4c8a-b89f-0bf3f6612586


In [13]:
# Optimizing optuna study for K-Nearest Neighbors
study_KNN.optimize(lambda trial: objective(trial, X_train, y_train, model='K-Nearest Neighbors'),n_trials=100)

[I 2026-06-12 23:58:14,449] Trial 0 finished with value: 0.5720073367425262 and parameters: {'knn_n_neighbors': 18, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 3}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:14,566] Trial 1 finished with value: 0.46806975908339493 and parameters: {'knn_n_neighbors': 4, 'knn_weights': 'uniform', 'knn_metric': 'minkowski', 'knn_p': 2}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:14,672] Trial 2 finished with value: 0.517102079400695 and parameters: {'knn_n_neighbors': 3, 'knn_weights': 'distance', 'knn_metric': 'euclidean', 'knn_p': 3}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:14,790] Trial 3 finished with value: 0.5440783447200765 and parameters: {'knn_n_neighbors': 12, 'knn_weights': 'uniform', 'knn_metric': 'euclidean', 'knn_p': 3}. Best is trial 0 with value: 0.5720073367425262.
[I 2026-06-12 23:58:15,028] Trial 4 finished with value: 0.6002708044687809 and p

In [14]:
print("Best score for KNN:", study_KNN.best_value)
print("Best params for KNN:", study_KNN.best_params)

Best score for KNN: 0.6031276199829867
Best params for KNN: {'knn_n_neighbors': 47, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 2}


In [15]:
# Fitting K-Nearest Neighbors pipeline with the best trial
pipe_KNN=build_pipeline(trial=study_KNN.best_trial, model= 'K-Nearest Neighbors')
fitted_pipe_KNN=pipe_KNN.fit(X_train, y_train)

### Support Vector Machine

In [18]:
# Creating optuna study for Support Vector Machine
study_SVM = optuna.create_study(direction="maximize")

[I 2026-06-05 19:47:54,790] A new study created in memory with name: no-name-180d694f-a739-4ec2-a529-07f1bd4cc500


In [19]:
# Optimizing optuna study for Support Vector Machine
study_SVM.optimize(lambda trial: objective(trial, X_train, y_train, model='Support Vector Machine'),n_trials=100)

[I 2026-06-05 19:47:57,085] Trial 0 finished with value: 0.596526081066813 and parameters: {'svc_C': 0.045226555476095345, 'svc_kernel': 'linear', 'svc_gamma': 6.4627416431187e-05, 'svc_degree': 4}. Best is trial 0 with value: 0.596526081066813.
[I 2026-06-05 19:48:02,080] Trial 1 finished with value: 0.6074161706691956 and parameters: {'svc_C': 0.002449641166797178, 'svc_kernel': 'rbf', 'svc_gamma': 0.04536150494034136, 'svc_degree': 2}. Best is trial 1 with value: 0.6074161706691956.
[I 2026-06-05 19:48:16,768] Trial 2 finished with value: 0.5233796065965034 and parameters: {'svc_C': 0.007682607615114961, 'svc_kernel': 'poly', 'svc_gamma': 1.28925952540971, 'svc_degree': 4}. Best is trial 1 with value: 0.6074161706691956.
[I 2026-06-05 19:48:19,296] Trial 3 finished with value: 0.621037937087001 and parameters: {'svc_C': 0.0012656583498744595, 'svc_kernel': 'poly', 'svc_gamma': 0.3364510615205489, 'svc_degree': 2}. Best is trial 3 with value: 0.621037937087001.
[I 2026-06-05 19:48:24

In [ ]:
print("Best score for SVM:", study_SVM.best_value)
print("Best params for SVM:", study_SVM.best_params)

In [ ]:
# Fitting Support Vector Machine pipeline with the best trial
pipe_SVM=build_pipeline(trial=study_SVM.best_trial, model= 'Support Vector Machine')
fitted_pipe_SVM=pipe_SVM.fit(X_train, y_train)

### Decision Tree

In [16]:
# Creating optuna study for Decision Tree
study_DT = optuna.create_study(direction="maximize")

[I 2026-06-12 23:58:37,414] A new study created in memory with name: no-name-ac1c3189-fc0e-4080-abee-1d8a03c6188d


In [17]:
# Optimizing optuna study for Decision Tree
study_DT.optimize(lambda trial: objective(trial, X_train, y_train, model='Decision Tree'),n_trials=100)

[I 2026-06-12 23:58:37,489] Trial 0 finished with value: 0.580203628528124 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 38, 'dt_min_samples_split': 20, 'dt_min_samples_leaf': 3, 'dt_max_features': 'sqrt'}. Best is trial 0 with value: 0.580203628528124.
[I 2026-06-12 23:58:37,567] Trial 1 finished with value: 0.5975765251262065 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 33, 'dt_min_samples_split': 19, 'dt_min_samples_leaf': 7, 'dt_max_features': 'log2'}. Best is trial 1 with value: 0.5975765251262065.
[I 2026-06-12 23:58:37,634] Trial 2 finished with value: 0.5800631849833937 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 38, 'dt_min_samples_split': 6, 'dt_min_samples_leaf': 5, 'dt_max_features': 'log2'}. Best is trial 1 with value: 0.5975765251262065.
[I 2026-06-12 23:58:37,704] Trial 3 finished with value: 0.5735688017430028 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 16, 'dt_min_samples_split': 13, 'dt_min_samples_leaf': 6, 'dt_m

In [18]:
print("Best score for DT:", study_DT.best_value)
print("Best params for DT:", study_DT.best_params)

Best score for DT: 0.6142241404458695
Best params for DT: {'dt_criterion': 'gini', 'dt_max_depth': 3, 'dt_min_samples_split': 20, 'dt_min_samples_leaf': 9, 'dt_max_features': None}


In [19]:
# Fitting Decision Tree pipeline with the best trial
pipe_DT=build_pipeline(trial=study_DT.best_trial, model= 'Decision Tree')
fitted_pipe_DT=pipe_DT.fit(X_train, y_train)

### Random Forest

In [20]:
# Creating optuna study for Random Forest
study_RF = optuna.create_study(direction="maximize")

[I 2026-06-12 23:58:44,964] A new study created in memory with name: no-name-6a62cebe-51d5-4bf3-a506-67877b3d6064


In [21]:
# Optimizing optuna study for Random Forest
study_RF.optimize(lambda trial: objective(trial, X_train, y_train, model='Random Forest'),n_trials=100)

[I 2026-06-12 23:58:51,044] Trial 0 finished with value: 0.5798547695110595 and parameters: {'rf_n_estimators': 1000, 'rf_criterion': 'gini', 'rf_max_depth': 10, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 10, 'rf_max_features': None, 'rf_bootstrap': False}. Best is trial 0 with value: 0.5798547695110595.
[I 2026-06-12 23:58:52,878] Trial 1 finished with value: 0.635609807203499 and parameters: {'rf_n_estimators': 900, 'rf_criterion': 'gini', 'rf_max_depth': 46, 'rf_min_samples_split': 2, 'rf_min_samples_leaf': 10, 'rf_max_features': 'sqrt', 'rf_bootstrap': False}. Best is trial 1 with value: 0.635609807203499.
[I 2026-06-12 23:58:58,725] Trial 2 finished with value: 0.5267264432495887 and parameters: {'rf_n_estimators': 600, 'rf_criterion': 'entropy', 'rf_max_depth': 39, 'rf_min_samples_split': 12, 'rf_min_samples_leaf': 1, 'rf_max_features': None, 'rf_bootstrap': False}. Best is trial 1 with value: 0.635609807203499.
[I 2026-06-12 23:58:59,639] Trial 3 finished with value: 0.6

In [22]:
print("Best score for RF:", study_RF.best_value)
print("Best params for RF:", study_RF.best_params)

Best score for RF: 0.6398899546223423
Best params for RF: {'rf_n_estimators': 700, 'rf_criterion': 'gini', 'rf_max_depth': 43, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 9, 'rf_max_features': 'log2', 'rf_bootstrap': True}


In [23]:
# Fitting Random Forest pipeline with the best trial
pipe_RF=build_pipeline(trial=study_RF.best_trial, model= 'Random Forest')
fitted_pipe_RF=pipe_RF.fit(X_train, y_train)

### Model performances

In [14]:
# Evaluate model performance
metrics_XGB=evaluate_model(fitted_pipe_XGB,X_test,y_test)
print(metrics_XGB['report'])

2026-08-11 10:28:16,304 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for XGBClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.89      0.80      0.84      1033
       Churn       0.57      0.72      0.64       374

    accuracy                           0.78      1407
   macro avg       0.73      0.76      0.74      1407
weighted avg       0.80      0.78      0.79      1407



In [16]:
metrics_XGB

{'classifier_name': 'XGBClassifier',
 'accuracy': 0.7810945273631841,
 'roc_auc': 0.8532491937195541,
 'f1': 0.6376470588235295,
 'precision': 0.569327731092437,
 'recall': 0.7245989304812834,
 'average_precision': 0.6670458317629006,
 'confusion_matrix': array([[828, 205],
        [103, 271]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.89      0.80      0.84      1033\n       Churn       0.57      0.72      0.64       374\n\n    accuracy                           0.78      1407\n   macro avg       0.73      0.76      0.74      1407\nweighted avg       0.80      0.78      0.79      1407\n'}

In [ ]:
metrics_LGBM=evaluate_model(fitted_pipe_LGBM,X_test,y_test)
print(metrics_LGBM['report'])

In [12]:
metrics_LR=evaluate_model(fitted_pipe_LR,X_test,y_test)
metrics_LR

2026-08-20 14:05:53,117 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for LogisticRegression classification model.


{'classifier_name': 'LogisticRegression',
 'accuracy': 0.7533759772565742,
 'roc_auc': 0.8516754585315602,
 'f1': 0.6312433581296493,
 'precision': 0.5238095238095238,
 'recall': 0.7941176470588235,
 'average_precision': 0.6578803710743749,
 'confusion_matrix': array([[763, 270],
        [ 77, 297]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.91      0.74      0.81      1033\n       Churn       0.52      0.79      0.63       374\n\n    accuracy                           0.75      1407\n   macro avg       0.72      0.77      0.72      1407\nweighted avg       0.81      0.75      0.77      1407\n'}

In [13]:
metrics_KNN=evaluate_model(fitted_pipe_KNN,X_test,y_test)
metrics_KNN

NameError: name 'fitted_pipe_KNN' is not defined

In [ ]:
metrics_SVM=evaluate_model(fitted_pipe_SVM,X_test,y_test)
print(metrics_SVM['report'])

In [26]:
metrics_DT=evaluate_model(fitted_pipe_DT,X_test,y_test)
metrics_DT

2026-06-13 00:01:34,157 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for DecisionTreeClassifier classification model.


{'classifier_name': 'DecisionTreeClassifier',
 'accuracy': 0.7455579246624022,
 'roc_auc': 0.825116088853917,
 'f1': 0.6183368869936035,
 'precision': 0.5141843971631206,
 'recall': 0.7754010695187166,
 'average_precision': 0.5724740321549325,
 'confusion_matrix': array([[759, 274],
        [ 84, 290]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.90      0.73      0.81      1033\n       Churn       0.51      0.78      0.62       374\n\n    accuracy                           0.75      1407\n   macro avg       0.71      0.76      0.71      1407\nweighted avg       0.80      0.75      0.76      1407\n'}

In [27]:
metrics_RF=evaluate_model(fitted_pipe_RF,X_test,y_test)
metrics_RF

2026-06-13 00:01:34,455 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for RandomForestClassifier classification model.


{'classifier_name': 'RandomForestClassifier',
 'accuracy': 0.7647476901208244,
 'roc_auc': 0.8511344870606872,
 'f1': 0.6301675977653631,
 'precision': 0.5412667946257198,
 'recall': 0.7540106951871658,
 'average_precision': 0.6636940545214198,
 'confusion_matrix': array([[794, 239],
        [ 92, 282]]),
 'report': '              precision    recall  f1-score   support\n\n   Not Churn       0.90      0.77      0.83      1033\n       Churn       0.54      0.75      0.63       374\n\n    accuracy                           0.76      1407\n   macro avg       0.72      0.76      0.73      1407\nweighted avg       0.80      0.76      0.78      1407\n'}